# middle_ai — 위험 감지·대응 모델 학습·테스트 (Colab)

**시작 전:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`

| 무엇 | 어디 |
|---|---|
| 코드·설정·팀 데이터(`data/custom`) | GitHub `chaejinseong/middle_ai` → 매번 최신으로 내려받음 |
| 학습 모델(`models/`)·결과(`outputs/`)·AI Hub 원본(`data/raw/`) | 구글 드라이브 `middle_ai_results/` → 세션이 끊겨도 유지 |
| 사전학습 모델(KcELECTRA, Qwen3Guard, Kanana…) | Colab 임시 디스크 → 맥·드라이브 용량 사용 안 함 |

**비공개 저장소라면:** 왼쪽 🔑(보안 비밀)에 `GH_TOKEN`(GitHub Personal Access Token, repo 읽기 권한) 등록

## 0. 준비

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠ GPU 없음 — 런타임 유형을 T4 GPU로 바꾸세요"

In [ ]:
import os, shutil, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = 'chaejinseong/middle_ai'
BRANCH = 'main'
CODE = '/content/middle_ai'
RESULTS = '/content/drive/MyDrive/middle_ai_results'      # 결과 저장 위치 (드라이브)

def secret(k):
    try: return userdata.get(k)
    except Exception: return None

# 1) GitHub에서 코드 받기 (있으면 최신으로 갱신)
token = secret('GH_TOKEN')
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if os.path.exists(CODE):
    subprocess.run(['git', '-C', CODE, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, CODE], check=True)
os.chdir(CODE)

# 2) models / outputs / data/raw 를 드라이브 폴더로 연결
for d in ['models', 'outputs', 'data/raw']:
    target = f'{RESULTS}/{d}'
    os.makedirs(target, exist_ok=True)
    if os.path.islink(d):
        continue
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.symlink(target, d)

if secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret('HF_TOKEN')
!git log -1 --format='코드 버전: %h %s (%cr)'
print('결과 저장 위치:', RESULTS)

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
!python scripts/check_env.py

## 1. F6 위험도 판단
### 1-1. 규칙 기준선

In [ ]:
!python -m src.risk.evaluate --mode rules

### 1-2. KcELECTRA 학습 → 규칙·분류기·결합 비교
샘플 데이터(24문장)라 점수는 의미 없음 — 팀 데이터로 교체 후 다시 실행

In [ ]:
!python -m src.risk.train_classifier

In [ ]:
!python -m src.risk.evaluate --mode all

In [ ]:
# 오탐 함정 세트 대신 다른 테스트셋으로 평가하려면
# !python -m src.risk.evaluate --mode all --test data/custom/risk_scenarios.csv

## 2. F4 안전 필터 (게이트 A/B)
같은 `gate_b_responses.csv`로 세 모델 비교 → 차단율·과차단율·지연시간

In [ ]:
!python -m src.guard.run_guard --model qwen0.6b

In [ ]:
!python -m src.guard.run_guard --model qwen4b

Kanana Safeguard 8B는 T4(15GB)에 그대로 안 들어가서 **4bit**로 로드 (약 6GB)

In [ ]:
!python -m src.guard.run_guard --model kanana --load_in_4bit

## 3. F5 추세 악화 탐지

In [ ]:
!python -m src.trend.trend_flag --simulate 100

## 4. F2 개인정보 마스킹

In [ ]:
!python -m src.pii.mask "저 김민수인데요 010-1234-5678로 연락 주세요. 서울시 마포구 합정동 살아요"

## 5. 결과 확인
결과는 드라이브 `middle_ai_results/outputs/`, 학습 모델은 `middle_ai_results/models/risk_classifier/`에 저장됨

In [ ]:
import glob, pandas as pd
for f in sorted(glob.glob('outputs/**/*.csv', recursive=True))[-5:]:
    print(f)
!du -sh models/* outputs 2>/dev/null

## 6. (나중에) 학습한 모델을 Hugging Face 비공개 저장소에 올리기
서비스 서버에서 이름으로 바로 받아 쓰기 위함. HF_TOKEN(write 권한) 필요

In [ ]:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# m = AutoModelForSequenceClassification.from_pretrained('models/risk_classifier')
# t = AutoTokenizer.from_pretrained('models/risk_classifier')
# m.push_to_hub('본인HF아이디/mindbridge-risk-classifier', private=True)
# t.push_to_hub('본인HF아이디/mindbridge-risk-classifier', private=True)